In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
torch.cuda.is_available()

True

# Generate data

In [12]:
import random

NUM_ITEMS = 8  # item ids will be 1..NUM_ITEMS -- 0 stays reserved for padding
BASE_PRICE = {item_id: random.randint(500, 3000) for item_id in range(1, NUM_ITEMS + 1)}

def generate_customer(min_events=3, max_events=8, min_age=20, max_age=60):
    n_events = random.randint(min_events, max_events)
    age = random.randint(min_age, max_age)

    items, ages, prices = [], [], []
    for _ in range(n_events):
        item_id = random.randint(1, NUM_ITEMS)
        price = BASE_PRICE[item_id] * random.lognormvariate(0, 0.15)

        items.append(item_id)
        ages.append(age)
        prices.append(round(price, 2))

        age += random.randint(0, 3)  # customer ages a bit between purchases

    return items, ages, prices

def generate_dataset(n_customers=500, min_events=3, max_events=8):
    return [generate_customer(min_events, max_events) for _ in range(n_customers)]

In [15]:
samples = generate_dataset()
len(samples)

500

In [17]:
samples[0], samples[-1]

(([7, 7, 7, 6, 7, 3, 3],
  [20, 22, 23, 26, 29, 32, 33],
  [1003.75, 862.53, 880.65, 1348.38, 1169.3, 2485.12, 2218.96]),
 ([2, 4, 1], [22, 22, 24], [1373.08, 1543.64, 807.94]))

# Start!

In [5]:
import numpy as np
import pandas as pd

In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# Create Dataset/DataLoader with custom collate_fn

In [4]:
class SampleDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self, ):
        return len(self.samples)
    
    def __getitem__(self, idx):
        return self.samples[idx]

In [20]:
dataset = SampleDataset(samples=samples)
len(dataset), dataset[0], dataset[-1]

(500,
 ([7, 7, 7, 6, 7, 3, 3],
  [20, 22, 23, 26, 29, 32, 33],
  [1003.75, 862.53, 880.65, 1348.38, 1169.3, 2485.12, 2218.96]),
 ([2, 4, 1], [22, 22, 24], [1373.08, 1543.64, 807.94]))

In [25]:
def next_item_collate_fn(batch):
    """What we do is to create: x, y which x is seq 0 to max-1 and y is -1"""
    items, ages, prices, next_items, next_ages, next_prices = [], [], [], [], [], []
    for item, age, price in batch:
        items.append(item[:-1])
        ages.append(age[:-1])
        prices.append(price[:-1])
        next_items.append(item[-1])
        next_ages.append(age[-1])
        next_prices.append(price[-1])
    return (
        items, ages, prices, next_items, next_ages, next_prices
    )

In [26]:
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=None)
next_item_dataloader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=next_item_collate_fn)

In [32]:
items, ages, prices, next_items, next_ages, next_prices = next_item_collate_fn(batch=[dataset[i] for i in range(4)])

In [35]:
items, next_items

([[7, 7, 7, 6, 7, 3], [5, 1, 2, 3], [8, 1, 1, 1, 8, 4], [7, 6, 6, 5]],
 [3, 7, 6, 3])

In [38]:
dataset[2]

([8, 1, 1, 1, 8, 4, 6],
 [29, 32, 34, 34, 36, 39, 41],
 [2763.85, 806.54, 682.72, 796.32, 3152.19, 1444.24, 932.08])

In [39]:
batch = next(iter(next_item_dataloader))
batch

([[5, 8, 5, 3],
  [7, 8, 6],
  [2, 7, 2],
  [6, 5],
  [4, 4, 4, 5, 2],
  [2, 1, 5, 5, 3, 5, 7],
  [2, 8, 7],
  [8, 6, 5, 3, 4, 5],
  [7, 6, 1, 2, 7],
  [7, 8, 2, 1, 3],
  [2, 7, 6, 3],
  [3, 2, 6, 1, 3, 2],
  [4, 5, 5, 6, 6],
  [1, 4, 8, 3, 5],
  [2, 6, 4, 3, 6, 2],
  [8, 3, 2, 2, 7],
  [2, 4, 1, 6, 5, 6],
  [6, 1, 7, 2, 2, 2],
  [8, 1, 4, 5],
  [6, 4, 6, 3, 8, 5],
  [5, 2, 7],
  [6, 5, 8, 1, 3],
  [4, 8],
  [8, 8, 4],
  [4, 8],
  [5, 6],
  [5, 6, 3, 6, 1, 5, 4],
  [1, 7, 6, 5, 3, 3],
  [5, 8],
  [2, 4, 1],
  [4, 5],
  [5, 5, 2, 7, 1, 7]],
 [[39, 42, 42, 45],
  [35, 35, 37],
  [52, 53, 54],
  [22, 25],
  [37, 39, 39, 41, 42],
  [30, 33, 36, 37, 40, 42, 42],
  [38, 40, 42],
  [56, 58, 60, 61, 61, 64],
  [60, 63, 66, 68, 68],
  [59, 60, 60, 62, 62],
  [58, 58, 60, 60],
  [42, 42, 45, 45, 46, 46],
  [49, 49, 51, 51, 53],
  [37, 37, 37, 40, 43],
  [60, 60, 62, 65, 68, 71],
  [53, 53, 54, 57, 58],
  [39, 40, 41, 41, 43, 44],
  [53, 54, 57, 57, 60, 61],
  [21, 24, 27, 30],
  [23, 23, 23, 25,

In [41]:
items, ages, prices, next_item, next_age, next_price = batch

In [45]:
items[0], next_item[0]

([5, 8, 5, 3], 7)

In [56]:
class CategoricalEmbedding(nn.Module):
    def __init__(self, vocab_size:int, d_model:int):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, X):
        return self.embedding(X)

class ContinuousEmbedding(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.embedding = nn.Linear(1, d_model)

    def forward(self, X):
        return self.embedding(X)

In [57]:
d_model = 10
cat_emb = CategoricalEmbedding(10, d_model)
cont_emb = ContinuousEmbedding(d_model)

In [63]:
cat_test = torch.tensor([
    [1,2,3],
    [4,5,6],
    [7,8,9]
], dtype=torch.long)
cont_test = torch.tensor([
    [1,2,3],
    [4,5,6],
    [7,8,9]
], dtype=torch.float)

In [64]:
cat_emb(cat_test).shape

torch.Size([3, 3, 10])

In [65]:
cont_emb(cont_test.unsqueeze(-1)).shape

torch.Size([3, 3, 10])

# what we need next is to master collate function